# Rigorous Project Re-evaluation (Version 2.0 - BCE Edition)

This notebook implements the final, scientifically perfect evaluation logic:
1. **BCE Loss:** Replaced sub-optimal MSE with Binary Cross-Entropy for both Classical and Quantum models.
2. **Quantum Probability Mapping:** Quantum expectation values (Pauli-Z) in `[-1, 1]` are mapped to probabilities in `[0, 1]`.
3. **Fixed Pipeline:** Leak-free scaling and random buffer sampling.
4. **5-Seed Rigor:** Full statistical average for honest comparisons.

In [ ]:
import sys
sys.path.append('..')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, log_loss
from utils.data_utils import load_higgs

SEEDS = [42, 7, 123, 1, 99]
N_SAMPLES = 1000
N_FEATURES = 8
PATH = '../data/HIGGS.csv.gz'

## 1. Classical MLP (Proper BCE training)
The MLP natively uses log-loss.

In [ ]:
classical_aucs = []

print("Training Classical MLP (5 seeds)...")
for seed in SEEDS:
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, 1)
    )
    
    model = MLPClassifier(
        hidden_layer_sizes=(100, 50, 25), 
        max_iter=500, 
        solver='adam',
        early_stopping=True, 
        random_state=seed
    )
    model.fit(X_tr, y_tr)
    
    probs = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, probs)
    classical_aucs.append(auc)
    print(f"  Seed {seed} | Test AUC: {auc:.4f}")

c_mean, c_std = np.mean(classical_aucs), np.std(classical_aucs)
print(f"\nClassical Result: {c_mean:.4f} ± {c_std:.4f}")

## 2. Quantum VQC (Binary Cross-Entropy)
We map expectation values to probabilities and minimize the log-loss.

In [ ]:
dev = qml.device('lightning.qubit', wires=N_FEATURES)

@qml.qnode(dev, interface='autograd')
def circuit(weights, x):
    # 3 Layers of Data Re-uploading
    for l in range(3):
        for i in range(len(x)): qml.RY(x[i], wires=i)
        for q in range(len(x)): qml.Rot(*weights[l, q], wires=q)
        for q in range(len(x)): qml.CNOT(wires=[q, (q + 1) % len(x)])
    return qml.expval(qml.PauliZ(0))

def vqc_prob(w, x, b):
    # Map [-1, 1] expectation to [0, 1] probability
    expval = circuit(w, x)
    return (expval + b + 1.0) / 2.0

def bce_loss(weights, bias, X, y):
    probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
    # Clip for numerical stability
    probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
    return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))

def train_quantum_bce(X_train, y_train, seed):
    pnp.random.seed(seed)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (3, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    for epoch in range(40):
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), 32):
            Xb = X_train[idx][start:start+32]
            yb = y_train[idx][start:start+32].astype(float)
            
            def cost(w_, b_):
                return bce_loss(w_, b_, Xb, yb)
            
            w, b = opt.step(cost, w, b)
    return w, b

quantum_aucs = []
print("\nTraining Quantum VQC with BCE (5 seeds)...")
for seed in SEEDS:
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, np.pi)
    )
    
    w_f, b_f = train_quantum_bce(X_tr, y_tr, seed)
    
    test_probs = np.array([float(vqc_prob(w_f, x, b_f)) for x in X_te])
    auc = roc_auc_score(y_te, test_probs)
    quantum_aucs.append(auc)
    print(f"  Seed {seed} | Test AUC: {auc:.4f}")

q_mean, q_std = np.mean(quantum_aucs), np.std(quantum_aucs)
print(f"\nQuantum Result (BCE): {q_mean:.4f} ± {q_std:.4f}")

## 3. Final Truth Table
Comparison of Mean Test AUC using proper classification statistics.

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Classical MLP', 'Quantum VQC'],
    'Mean Test AUC': [c_mean, q_mean],
    'Std Dev': [c_std, q_std]
})
print(results_df)

plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['Mean Test AUC'], yerr=results_df['Std Dev'], 
        capsize=10, color=['#B4B2A9', '#7F77DD'])
plt.ylabel('Test AUC')
plt.title('Rigorous Evaluation (BCE Loss, N=1000)')
plt.ylim(0.5, 0.75)
plt.show()